<a href="https://colab.research.google.com/github/Rageel-28/244107020136-Machine-Learning/blob/main/JS05/JS05_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS05 - Klasifikasi 1
## Lab 3 - Klasifikasi Naive Bayes dengan Data Multinomial (SMS Spam)

**Tujuan:** melakukan klasifikasi teks SMS menjadi **spam** atau **ham** menggunakan **Multinomial Naive Bayes** dengan fitur *Bag of Words* (`CountVectorizer`).

Nilai multinomial adalah nilai hasil proses **menghitung** (counting), contoh klasiknya jumlah kemunculan kata pada klasifikasi teks.

**Dataset:** `spam.csv` - spam adalah SMS sampah, ham adalah SMS biasa.

## Langkah 0 - Persiapan Dataset

In [1]:
# Upload dataset (khusus Google Colab).
# Jika file sudah ada di folder kerja, langkah upload dilewati.
import os

if not os.path.exists('spam.csv'):
    try:
        from google.colab import files
        print('Silakan upload file spam.csv')
        files.upload()
    except ImportError:
        raise FileNotFoundError('spam.csv tidak ditemukan. Letakkan file di folder yang sama dengan notebook ini.')

print('File spam.csv tersedia:', os.path.exists('spam.csv'))

File spam.csv tersedia: True


## Langkah 1 - Load Data
Memuat data ke DataFrame lalu inspeksi sederhana untuk memastikan apakah perlu pra pengolahan sebelum ekstraksi fitur dan pemodelan.

> `encoding='latin-1'` diperlukan karena data tidak menggunakan UTF-8.

In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv('spam.csv', encoding='latin-1')  # spesifikasi encoding diperlukan karena data tidak menggunakan UTF-8

df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


Terdapat 3 kolom yang tidak bermanfaat (`Unnamed: 2`, `Unnamed: 3`, `Unnamed: 4`), sehingga perlu dibuang. Selain itu kolom **v1** (label) dan **v2** (teks SMS) akan diganti namanya agar mudah dibaca.

## Langkah 2 - Preprocessing
Yang dilakukan pada tahap ini:

1. Drop kolom yang tidak digunakan
2. Ubah nama kolom v1 (label) dan v2 (teks SMS)
3. Inspeksi data
4. Encode label
5. Memisahkan fitur dengan label

### Langkah 2a - Drop Kolom

In [3]:
# Drop 3 kolom terakhir (kolom index ke-2 dan seterusnya)
df = df.drop(columns=df.columns[2:])

# Cek data
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### Langkah 2b - Ubah Nama Kolom
Kolom `v1` menjadi `Labels` dan `v2` menjadi `SMS` (nama ini dipakai pada langkah-langkah berikutnya).

In [4]:
df = df.rename(columns={'v1': 'Labels', 'v2': 'SMS'})

# Cek data
df.head()

,Labels,SMS
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### Langkah 2c - Inspeksi Data

In [5]:
# Cek Jumlah Data Per Kelas
print(df['Labels'].value_counts())
print('\n')

# Cek Kelengkapan Data
print(df.info())
print('\n')

# Cek Statistik Deskriptif
print(df.describe())

Labels
ham     4825
spam     747
Name: count, dtype: int64


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Labels  5572 non-null   object
 1   SMS     5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB
None


       Labels                     SMS
count    5572                    5572
unique      2                    5169
top       ham  Sorry, I'll call later
freq     4825                      30


### Langkah 2d - Encoding Label
Label teks diubah menjadi angka: `spam = 1`, `ham = 0`.

In [6]:
# Data untuk label
new_labels = {
    'spam': 1,
    'ham': 0
}

# Encode label
df['Labels'] = df['Labels'].map(new_labels)

# Cek data
df.head()

,Labels,SMS
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


### Langkah 2e - Pisahkan Fitur dengan Label

In [7]:
X = df['SMS'].values
y = df['Labels'].values

print('Jumlah teks (X):', len(X))
print('Jumlah label (y):', len(y))

Jumlah teks (X): 5572
Jumlah label (y): 5572


## Langkah 3 - Ekstraksi Fitur
Ekstraksi fitur setiap SMS menggunakan konsep **Bag of Words** melalui `CountVectorizer`.

Untuk mencegah **information leakage**, data **di-split terlebih dahulu** (80% latih, 20% uji), baru kemudian transformasi dilakukan:

- Data latih: `fit_transform` (belajar kosakata sekaligus mengubah menjadi vektor)
- Data uji: hanya `transform` (agar data uji tetap asing bagi model dan tidak ikut membentuk kosakata)

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# Split data training dan testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

# Inisiasi CountVectorizer
bow = CountVectorizer()

# Fitting dan transform X_train dengan CountVectorizer
X_train = bow.fit_transform(X_train)

# Transform X_test
# Mengapa hanya transform? Kita tidak menginginkan model mengetahui parameter
# yang digunakan CountVectorizer untuk fitting data X_train.
# Sehingga, data testing tetap menjadi data yang asing bagi model nantinya
X_test = bow.transform(X_test)

Cek fitur hasil `CountVectorizer`.

In [9]:
print(len(bow.get_feature_names_out()))
print(f'Dimensi data: {X_train.shape}')

7727
Dimensi data: (4457, 7727)


## Langkah 4 - Training dan Evaluasi Model
Menggunakan algoritma **Multinomial Naive Bayes** (`MultinomialNB`), kemudian dievaluasi dengan akurasi pada data latih dan data uji.

In [10]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Inisiasi MultinomialNB
mnb = MultinomialNB()

# Fit model
mnb.fit(X_train, y_train)

# Prediksi dengan data training
y_pred_train = mnb.predict(X_train)

# Evaluasi akurasi data training
acc_train = accuracy_score(y_train, y_pred_train)

# Prediksi dengan data testing
y_pred_test = mnb.predict(X_test)

# Evaluasi akurasi data testing
acc_test = accuracy_score(y_test, y_pred_test)

# Print hasil evaluasi
print(f'Hasil akurasi data train: {acc_train}')
print(f'Hasil akurasi data test: {acc_test}')

Hasil akurasi data train: 0.9946152120260264
Hasil akurasi data test: 0.9775784753363229


## Kesimpulan Lab 3

- Teks SMS diubah menjadi vektor *Bag of Words* dengan `CountVectorizer` (tanpa stop words), menghasilkan **7727 fitur (kata unik)** dari 4457 SMS latih.
- Model Multinomial Naive Bayes memperoleh **akurasi latih 0.9946** dan **akurasi uji 0.9776**. Selisih keduanya kecil (sekitar 1.7%), artinya model tidak mengalami overfitting yang berarti.
- Perlu diingat kelas tidak seimbang (ham 4825, spam 747). Model yang selalu menebak "ham" saja sudah akan mendapat akurasi sekitar 86.6%, jadi akurasi saja belum cukup; evaluasi precision, recall, dan f1-score untuk kelas spam dilakukan pada **Tugas Lab 2**.